Assignment 28: Q&A RAG Chatbot with Message History


In [1]:
#Task 1 - Install Libraries
!pip install -q langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface langchain-groq sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
#Task 1 - Load PDF
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("DevAnand_Resume.pdf")
documents = loader.load()
print("Documents:", len(documents))
print(documents[0].page_content[:500])

/tmp/ipykernel_506/2714394057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Documents: 1
DEV ANAND J 
+91 99524 72762  |  devanand3254@gmail.com  |  GitHub  |  LinkedIn  |  Portfolio 
EDUCATION 
Karpagam Institute of Technology — B.Tech in Information Technology                                                           Nov 2022 – May 2026 
PROFESSIONAL EXPERIENCE 
VSeek Ventures — Backend Developer Intern | Hybrid                                                                                       July 2025 – May 2026 
• Built REST APIs using Go, Echo, and PostgreSQL with JWT authe


In [3]:
#Task 2 - Text Splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print("Chunks:", len(chunks))
print(chunks[0].page_content)

Chunks: 20
DEV ANAND J 
+91 99524 72762  |  devanand3254@gmail.com  |  GitHub  |  LinkedIn  |  Portfolio 
EDUCATION 
Karpagam Institute of Technology — B.Tech in Information Technology                                                           Nov 2022 – May 2026 
PROFESSIONAL EXPERIENCE


In [4]:
import sys

print(sys.executable)
print(sys.version)

/usr/bin/python3
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [5]:
# Task 3 - Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
test = embedding_model.embed_query(chunks[0].page_content)
print("Embedding size:", len(test))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding size: 384


In [6]:
# Task 4 - FAISS Retriever
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(chunks, embedding_model)
retriever = vector_store.as_retriever(search_kwargs={"k":3})
docs = retriever.invoke("What projects has Dev Anand worked on?")
print(docs[0].page_content)

DEV ANAND J 
+91 99524 72762  |  devanand3254@gmail.com  |  GitHub  |  LinkedIn  |  Portfolio 
EDUCATION 
Karpagam Institute of Technology — B.Tech in Information Technology                                                           Nov 2022 – May 2026 
PROFESSIONAL EXPERIENCE


In [7]:
# Task 5 - Prompt
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
("system","Answer only from the provided context. If the answer is not available, reply 'I don't know'.\n\nContext:\n{context}"),
MessagesPlaceholder(variable_name="chat_history"),
("human","{question}")])
print("Prompt Ready")

Prompt Ready


In [16]:
# Task 6 - Groq LLM
import os
from getpass import getpass
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

os.environ["GROQ_API_KEY"] = getpass("Enter GROQ API Key: ")
llm = ChatGroq(model="llama-3.3-70b-versatile",temperature=0)
output_parser = StrOutputParser()
chat_history=[]
print("Groq Ready")

Enter GROQ API Key: ··········
Groq Ready


In [17]:
# Task 7 - Message History

from langchain_core.messages import HumanMessage, AIMessage
def ask_question(question):
    docs = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in docs])

    messages = prompt.invoke({"context":context,"question":question,"chat_history":chat_history})
    response = llm.invoke(messages)
    answer = output_parser.invoke(response)
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))
    print("\nQuestion:",question)
    print("\nAnswer:",answer)

In [18]:
MAX_MESSAGES=6
def trim_history():
    global chat_history
    if len(chat_history)>MAX_MESSAGES:
        chat_history=chat_history[-MAX_MESSAGES:]
    print("History:",len(chat_history))

In [19]:
# Task 9 - Testing
tests=[
"What projects has Dev Anand worked on?",
"Tell me more about VisionChallan AI.",
"What certifications does he have?",
"Explain the previous answer.",
"Does the resume mention Kubernetes?"]
for q in tests:
    ask_question(q)
    trim_history()


Question: What projects has Dev Anand worked on?

Answer: I don't know. The provided context does not mention any specific projects that Dev Anand has worked on, but it does mention that he presented a research paper called 'Tech Career Navigator' at an international conference.
History: 2

Question: Tell me more about VisionChallan AI.

Answer: VisionChallan AI is an Intelligent Traffic Violation Detection and Challan Management System. It is built using YOLOv8 and deep learning to detect helmet non-compliance in real-time traffic monitoring. That's all the information available about VisionChallan AI from the provided context.
History: 4

Question: What certifications does he have?

Answer: Dev Anand has the following certifications:

1. AWS Certified Cloud Practitioner
2. AWS Certified Solutions Architect – Associate
3. Salesforce Certified Agentforce Specialist
History: 6

Question: Explain the previous answer.

Answer: The previous answer was incorrect. The provided context does 

In [20]:
# Task 10 - Chatbot
chat_history=[]
print("Conversational RAG Chatbot")
print("Type exit to stop")
while True:
    q=input("You: ")
    if q.lower()=="exit":
        break
    ask_question(q)
    trim_history()

Conversational RAG Chatbot
Type exit to stop
You: python

Question: python

Answer: Python is mentioned as one of the programming languages used in the projects and is also listed under "TECHNICAL SKILLS" as a programming language. It was used in combination with various technologies such as FastAPI, YOLOv8, OpenCV, EasyOCR, PostgreSQL, MongoDB, Docker, REST APIs, NLP, and ML. Additionally, it was used with AWS services like Comprehend Medical, Lambda, API Gateway, and others.
History: 2
You: AWS

Question: AWS

Answer: AWS (Amazon Web Services) is mentioned as a cloud platform used in the projects. Specifically, the following AWS services are listed:

1. S3
2. Lambda
3. CloudWatch
4. Comprehend Medical
5. API Gateway
6. DynamoDB
7. Amplify
8. Personalize 

These services were used in combination with other technologies to build a Smart Healthcare Assistant, an AWS-powered AI healthcare system.
History: 4
You: backend

Question: backend

Answer: Backend is mentioned as the area of focu

## Task 11 - Observations

1. Conversational RAG uses chat history while normal RAG treats each question independently.

2. Message history helps answer follow-up questions.

3. Long history improves context but increases token usage.

4. Trimming keeps recent conversations and improves performance.
